# 03 — Binary Classification Filter Preprocessing
**Dataset:** Diabetic Retinopathy Resized (`tanlikesmath/diabetic-retinopathy-resized`)
**Goal:** Download via KaggleHub, preprocess fundus images with DR grade labels, save as `.npz`, analyze class distribution.
> Run on **Google Colab**. Set your Kaggle credentials before running.

## 1. Install Dependencies

In [ ]:
import subprocess, sys
subprocess.run([sys.executable, '-m', 'pip', '-q', 'install',
                'kagglehub', 'opencv-python-headless', 'tqdm', 'matplotlib', 'scikit-learn', 'numpy', 'pandas'],
               check=True)
print('Dependencies ready.')

## 2. Kaggle Authentication

In [ ]:
import os
from pathlib import Path

kaggle_dir = Path.home() / '.kaggle'
kaggle_dir.mkdir(parents=True, exist_ok=True)
kaggle_json = kaggle_dir / 'kaggle.json'

if not kaggle_json.exists() and 'google.colab' in sys.modules:
    from google.colab import files
    print('Upload your kaggle.json file:')
    uploaded = files.upload()
    if 'kaggle.json' in uploaded:
        kaggle_json.write_bytes(uploaded['kaggle.json'])
        kaggle_json.chmod(0o600)

# os.environ['KAGGLE_USERNAME'] = 'your_username'
# os.environ['KAGGLE_KEY'] = 'your_api_key'

if kaggle_json.exists():
    kaggle_json.chmod(0o600)
    print('Kaggle credentials ready.')
else:
    print('WARNING: kaggle.json not found.')

## 3. Repository Setup & Imports

In [ ]:
import sys
import shutil
import numpy as np
import pandas as pd
import cv2
import matplotlib.pyplot as plt
from pathlib import Path
from tqdm import tqdm
from sklearn.model_selection import train_test_split

repo_root = Path.cwd()
if not (repo_root / 'src').exists():
    repo_root = repo_root.parent
if str(repo_root / 'src') not in sys.path:
    sys.path.insert(0, str(repo_root / 'src'))

from engine.image_preprocessing import PreprocessConfig, preprocess_fundus_image, save_preprocessed

data_dir = repo_root / 'data'
raw_dir = data_dir / 'raw'
processed_dir = data_dir / 'processed'
raw_dir.mkdir(parents=True, exist_ok=True)
processed_dir.mkdir(parents=True, exist_ok=True)
print('Repo root:', repo_root)

## 4. Download DR Dataset via KaggleHub

In [ ]:
import kagglehub

DATASET_SLUG = 'tanlikesmath/diabetic-retinopathy-resized'
dr_root = raw_dir / 'dr_resized'
dr_root.mkdir(parents=True, exist_ok=True)

already_downloaded = any(dr_root.rglob('*.png')) or any(dr_root.rglob('*.jpeg')) or any(dr_root.rglob('*.csv'))
if not already_downloaded:
    print(f'Downloading {DATASET_SLUG} ...')
    download_path = Path(kagglehub.dataset_download(DATASET_SLUG))
    if download_path != dr_root:
        shutil.copytree(download_path, dr_root, dirs_exist_ok=True)
    print('Download complete:', dr_root)
else:
    print('Dataset already present at:', dr_root)

print(f'Total files: {len(list(dr_root.rglob("*")))}' )

## 5. Load Labels & Index Images

In [ ]:
IMAGE_EXTS = {'.png', '.jpg', '.jpeg'}

# Try to find CSV label file
csv_files = list(dr_root.rglob('*.csv'))
df = None
if csv_files:
    label_csv = csv_files[0]
    print(f'Found label file: {label_csv}')
    raw_df = pd.read_csv(label_csv)
    print('Columns:', list(raw_df.columns))
    print(raw_df.head())
    # Auto-detect image and label columns
    img_col = next((c for c in raw_df.columns if 'image' in c.lower() or 'file' in c.lower() or 'name' in c.lower()), raw_df.columns[0])
    lbl_col = next((c for c in raw_df.columns if 'level' in c.lower() or 'label' in c.lower() or 'grade' in c.lower() or 'class' in c.lower()), raw_df.columns[1])
    print(f'Using image col: "{img_col}", label col: "{lbl_col}"')
    all_imgs = list(dr_root.rglob('*'))
    img_map = {p.stem: p for p in all_imgs if p.suffix.lower() in IMAGE_EXTS}
    rows = []
    for _, row in raw_df.iterrows():
        stem = Path(str(row[img_col])).stem
        img_path = img_map.get(stem)
        if img_path:
            rows.append({'image_path': img_path, 'label': int(row[lbl_col]), 'binary_label': 0 if int(row[lbl_col]) == 0 else 1})
    df = pd.DataFrame(rows)
else:
    # Fallback: infer labels from folder structure
    print('No CSV found. Inferring labels from folder names...')
    rows = []
    for img_path in dr_root.rglob('*'):
        if img_path.suffix.lower() not in IMAGE_EXTS: continue
        try:
            label = int(img_path.parent.name)
        except ValueError:
            label = -1
        rows.append({'image_path': img_path, 'label': label, 'binary_label': 0 if label == 0 else 1})
    df = pd.DataFrame(rows)

print(f'\nTotal images indexed: {len(df)}')
print(df['label'].value_counts().sort_index())

## 6. Train / Val / Test Split (70/15/15) — Stratified

In [ ]:
if df.empty:
    raise RuntimeError('No images found — check dataset download.')

stratify_col = df['binary_label'] if df['binary_label'].nunique() > 1 else None
train_df, temp_df = train_test_split(df, test_size=0.30, random_state=42, stratify=stratify_col)
s2 = temp_df['binary_label'] if temp_df['binary_label'].nunique() > 1 else None
val_df, test_df = train_test_split(temp_df, test_size=0.50, random_state=42, stratify=s2)

print(f'Train: {len(train_df)} | Val: {len(val_df)} | Test: {len(test_df)}')
splits = {'train': train_df, 'val': val_df, 'test': test_df}

## 7. Batch Preprocessing & Save

In [ ]:
config = PreprocessConfig(target_size=(512, 512), normalization='zero_one')
DATASET_NAME = 'dr_resized'
errors = []

for split_name, split_df in splits.items():
    out_dir = processed_dir / DATASET_NAME / split_name
    out_dir.mkdir(parents=True, exist_ok=True)
    print(f'\nProcessing {split_name} ({len(split_df)} images)...')
    for _, row in tqdm(split_df.iterrows(), total=len(split_df), desc=split_name):
        try:
            result = preprocess_fundus_image(row['image_path'], config=config)
            out_file = out_dir / (row['image_path'].stem + f'_label{row["label"]}.npz')
            save_preprocessed(result, out_file)
        except Exception as e:
            errors.append({'file': str(row['image_path']), 'error': str(e)})
            print(f'  WARNING: skipped {row["image_path"].name} — {e}')

print(f'\nDone. Errors: {len(errors)}')

## 8. Dataset Statistics & Class Distribution

In [ ]:
print('=== Dataset Statistics ===')
for split_name, split_df in splits.items():
    print(f'{split_name}: {len(split_df)} images')

# DR Grade distribution bar chart
grade_names = {0: 'No DR', 1: 'Mild', 2: 'Moderate', 3: 'Severe', 4: 'Proliferative'}
counts = df['label'].value_counts().sort_index()
labels = [grade_names.get(i, f'Grade {i}') for i in counts.index]
colors = ['#2ecc71', '#f39c12', '#e67e22', '#e74c3c', '#8e44ad'][:len(counts)]

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# DR Grade (5-class)
ax1.bar(labels, counts.values, color=colors, edgecolor='black', linewidth=0.7)
ax1.set_title('DR Grade Distribution (5-class)', fontsize=12, fontweight='bold')
ax1.set_xlabel('DR Grade')
ax1.set_ylabel('Number of Images')
for i, v in enumerate(counts.values):
    ax1.text(i, v + max(counts.values)*0.01, str(v), ha='center', fontsize=9)

# Binary distribution (No DR vs DR)
binary_counts = df['binary_label'].value_counts().sort_index()
binary_labels = ['No DR (Grade 0)', 'DR (Grade 1-4)']
ax2.pie(binary_counts.values, labels=binary_labels, autopct='%1.1f%%',
        colors=['#2ecc71', '#e74c3c'], startangle=90, textprops={'fontsize': 11})
ax2.set_title('Binary Classification Distribution', fontsize=12, fontweight='bold')

plt.suptitle('Diabetic Retinopathy Dataset — Class Distribution', fontsize=13)
plt.tight_layout()
plt.show()

print(f'\nClass imbalance ratio (DR/No-DR): {binary_counts.get(1,0)/max(binary_counts.get(0,1),1):.2f}')

## 9. Visualization — Sample Images per Class

In [ ]:
grade_names = {0: 'No DR', 1: 'Mild', 2: 'Moderate', 3: 'Severe', 4: 'Proliferative'}
unique_labels = sorted(df['label'].unique())
n_cols = min(4, len(unique_labels))
n_rows = (len(unique_labels) + n_cols - 1) // n_cols

fig, axes = plt.subplots(n_rows, n_cols, figsize=(4 * n_cols, 4 * n_rows))
axes = np.array(axes).flatten()

for i, lbl in enumerate(unique_labels):
    subset = df[df['label'] == lbl]
    sample_row = subset.sample(1, random_state=42).iloc[0]
    img_bgr = cv2.imread(str(sample_row['image_path']))
    if img_bgr is not None:
        axes[i].imshow(cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB))
    axes[i].set_title(f'Grade {lbl}: {grade_names.get(lbl, "Unknown")}\n(n={len(subset)})', fontsize=9)
    axes[i].axis('off')

for j in range(i+1, len(axes)):
    axes[j].axis('off')

plt.suptitle('Sample Images per DR Grade', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

## Next Steps
- Feed `.npz` files from `data/processed/dr_resized/` into a classification model
- Recommended architecture: **EfficientNet-B4** or **ResNet-50** fine-tuned on fundus images
- Use this notebook as a **pre-filter**: classify No-DR vs DR before running segmentation
- Address class imbalance with: **weighted cross-entropy**, **oversampling (SMOTE)**, or **focal loss**
- Metrics: **AUC-ROC**, **F1-score**, **Sensitivity**, **Specificity** (clinical priority: high sensitivity)